# AI Attitudes Survey - Example Hypothesis Tests

This notebook contains example hypothesis tests using data from our AI Attitudes survey. Examples include:

1. Testing for a significant difference in population proportions.
2. Testing for a signficant difference in population means.

## Preliminaries

Run the code cell to load and clean the data and import necessary libraries. This code renames columns and drops unneeded columns.

Run this code **before** running any of the examples in this notebook.

In [2]:
from datetime import date
import numpy as np, pandas as pd
import matplotlib.pyplot as plt0
from textblob import TextBlob, Word
from wordcloud import WordCloud

file = "/home/shared/AI_Attitudes.csv"
survey = pd.read_csv(file)

# Rename and drop columns
survey.columns=['ID', 'Start', 'End', 'Email', 'Name', 'Last Edit', 'Age', 'Gender', 'Excited', 'Why Excited', 'Concerned', 'Why Concerned', 'Use Freq', 'How Used', 'Tools', 'Report']
survey.drop(columns=['Email', 'Name', 'Last Edit', 'Report'], inplace=True)
survey

# Some recoding using np.select()
# Firstly, create a Net Excited-Concerned score (Net E-C) based on Excited - Concerned
# Then recode age into ascending numerical values so we can order output later
survey['Net E-C'] = survey['Excited'] - survey['Concerned']
conditions = [survey['Net E-C']>0, survey['Net E-C']==0, survey['Net E-C']<0]
values = ['More excited than concerned', 'Equally excited and concerned', 'More concerned than excited']
survey['Net E-C Cat'] = np.select(conditions, values, default='')
conditions = [survey['Age']=='Under 18', survey['Age']=='18-24', survey['Age']=='25-34', survey['Age']=='35-44', survey['Age']=='45-54', survey['Age']=='55 or over', survey['Age']=='Prefer not to say']
values = [0, 1, 2, 3, 4, 5, 6]
survey['Age Code'] = np.select(conditions, values, default=0)
survey

,ID,Start,End,Age,Gender,Excited,Why Excited,Concerned,Why Concerned,Use Freq,How Used,Tools,Net E-C,Net E-C Cat,Age Code
0,11,2/20/26 9:26:30,2/20/26 9:28:20,45-54,Male,4,Making coding easier,9,Concerned about impact on employment prospects...,Several times a day,"search, finding out how to code things.",Microsoft Copilot;Chat GPT;Google Gemini;,-5,More concerned than excited,4
1,12,2/20/26 9:52:31,2/20/26 10:03:38,45-54,Female,8,It simplifies otherwise manual tasks and it’s ...,8,It has become a crutch for me - I have come to...,Several times a day,See above.,Chat GPT;Claude (Anthropic);Microsoft Copilot;,0,Equally excited and concerned,4
2,13,2/20/26 11:52:56,2/20/26 11:54:14,55 or over,Female,7,NaN,4,NaN,A few times a week,Answer search questions,Chat GPT;Microsoft Copilot;,3,More excited than concerned,5
3,14,2/20/26 11:55:10,2/20/26 12:06:45,25-34,Male,10,The possibilities that become more probable wh...,10,Dependency on AI to solve simple problems we s...,Several times a day,"Resume, Explain Complex Code, Math Tutor, draf...",Notebook LM;Google Gemini;Perplexity AI;Claude...,0,Equally excited and concerned,2
4,15,2/20/26 13:38:13,2/20/26 13:39:51,45-54,Female,2,NaN,8,"Outsourced thinking, loss of human creativity ...",A few times a week,Analyse data.,Notebook LM;Chat GPT;Microsoft Copilot;,-6,More concerned than excited,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
108,119,3/4/26 10:13:23,3/4/26 10:15:23,25-34,Female,8,Being able to use AI as a tool to further my k...,4,The movie IRobot,Several times a day,"Spell check homework, answer questions",Google Gemini;Grammarly;Claude (Anthropic);Cha...,4,More excited than concerned,2
109,120,3/4/26 11:00:36,3/4/26 11:18:28,25-34,Female,5,I feel neutral about the use of AI in people’s...,6,"Up until about 3 weeks ago, I did not think of...",A few times a week,I actually use it mainly for health reasons. I...,Chat GPT;Google Gemini;,-1,More concerned than excited,2
110,121,3/4/26 11:19:28,3/4/26 11:23:29,45-54,Female,8,I can do work faster since i write many emails.,9,Water usage and young people using there brain...,About once a day,Help with items at work!,Chat GPT;Microsoft Copilot;Google Gemini;,-1,More concerned than excited,4
111,122,3/4/26 11:24:48,3/4/26 11:29:46,35-44,Female,8,I like to use AI for career development as wel...,6,I do worry about how it affects the environment.,Several times a day,"I use it for resume, cover letters, career pla...",Chat GPT;Perplexity AI;Google Gemini;,2,More excited than concerned,3


## 1. Testing for Significant Difference in Proportions

In this example we compare the proportion of males and females who reported to be 'more concerned than excited' based on their net excited-concerned score.

We are interested in whether any difference in proportions for these two groups is statistically significant. We will use a 5% significance level, which corresponds to a 95% confidence level.

We will use $p_m$ to represent the proportion of males who reported being more concerned than excited. We will use $p_f$ to represent the proportion of males who reported being more concerned than excited.

Our hypotheses are:

$$ H_0: p_m = p_f$$
$$ H_1: p_m \ne p_f$$

Significance level: $\alpha = 0.05$

In [3]:
# First compare counts for males and females
print("Male")
print(survey[survey['Gender']=='Male']['Net E-C Cat'].value_counts())
print("Female")
print(survey[survey['Gender']=='Female']['Net E-C Cat'].value_counts())

Male
Net E-C Cat
More concerned than excited      20
More excited than concerned      20
Equally excited and concerned    10
Name: count, dtype: int64
Female
Net E-C Cat
More concerned than excited      37
More excited than concerned      18
Equally excited and concerned     6
Name: count, dtype: int64


In [4]:
# Now conduct the hypothesis test
from statsmodels.stats.proportion import proportions_ztest

more_concerned_m = 20
more_concerned_f = 37
total_m = 50
total_f = 61

print(f"Percent of males more concerned than excited: {more_concerned_m/total_m*100:.1f}%")
print(f"Percent of females more more concerned than excited: {more_concerned_f/total_f*100:.1f}%")

count = np.array([more_concerned_m, more_concerned_f])
nobs = np.array([total_m,total_f])

# Perform the z-test for two proportions
z_stat, p_value = proportions_ztest(count=count, nobs=nobs, alternative='two-sided')

print(f"Z-statistic: {z_stat:.4f}")
print(f"P-value: {p_value:.4f}")

# Decision
if p_value < 0.05:
    print("Reject the null hypothesis: Significant difference found at the 95% confidence level.")
else:
    print("Fail to reject the null hypothesis: No significant difference found at the 95% confidence level.")

Percent of males more concerned than excited: 40.0%
Percent of females more more concerned than excited: 60.7%
Z-statistic: -2.1663
P-value: 0.0303
Reject the null hypothesis: Significant difference found at the 95% confidence level.


## 2. Testing for Significant Difference in Means

In this example we compare the mean 'excited' score for those aged under 35 with all others. 

We are interested in whether any difference in the mean 'excited score' for these two groups is statistically significant. We will use a 5% significance level, which corresponds to a 95% confidence level.

We will use $\mu_1$ to represent the mean excited score for those under 35. We will use $\mu_2$ to represent the mean excited score for others.

Our hypotheses are:

$$ H_0: \mu_1 = \mu_2$$
$$ H_1: \mu_1 \ne \mu_2$$

Significance level: $\alpha = 0.05$

In [5]:
# Let's print the means() to see if they are similar
print(f"Mean excited score for those under 35: {survey[survey['Age Code'] < 2]['Excited'].mean():.2f}")
print(f"Mean excited score for others: {survey[survey['Age Code'] >= 2]['Excited'].mean():.2f}")

Mean excited score for those under 35: 6.03
Mean excited score for others: 5.54


In [6]:
from scipy import stats

t1 = survey[survey['Age Code'] < 2]['Excited']
t2 = survey[survey['Age Code'] >= 2]['Excited']

# Perform paired samples t-test
t_statistic_rel, p_value_rel = stats.ttest_ind(t1, t2)

print(f"Independent Samples T-test Results")
print(f"T-statistic: {t_statistic_rel:.4f}")
print(f"P-value: {p_value_rel:.4f}\n")
if p_value_rel < 0.05:
    print(f"Since {p_value_rel:.4f} < 0.05, the difference between the means is statistically significant at the 95% confidence level.")
else:
    print(f"Since {p_value_rel:.4f} >= 0.05, the difference between the means is NOT statistically significant at the 95% confidence level.")

Independent Samples T-test Results
T-statistic: 0.8459
P-value: 0.3994

Since 0.3994 >= 0.05, the difference between the means is NOT statistically significant at the 95% confidence level.


## Challenge Task

Can you modify the code in the first example to test if there is a significant difference between males and females in the proportion who are 'more excited than concerned'?

In [ ]:
# Type your code here
